In [ ]:
# =========================================================
# 02_era5_gee_extracao.ipynb - EXTRAÇÃO INMET + ERA5
# =========================================================
# Objetivo:
# - Autenticar e inicializar no Google Earth Engine (GEE)
# - Extrair precipitação mensal acumulada do ERA5
# - Combinar com os dados do INMET
# =========================================================

# =========================================================
# 0) Instalar e importar bibliotecas
# =========================================================
!pip install earthengine-api geemap --quiet

import ee
import pandas as pd
import numpy as np
from google.colab import files

# =========================================================
# 1) Autenticação e inicialização no GEE
# =========================================================
ee.Authenticate()
ee.Initialize(project='DEFINA AQUI')  #DEFINIR PROJETO DO GEE 

In [ ]:
# =========================================================
# 2) Configurações do período de estudo
# =========================================================
ANO_INICIAL = 1980
ANO_FINAL = 2023
LIMITE_PERCENTUAL = 90  # Filtrar estações com ≥90% de dados

# =========================================================
# 3) Ler base INMET filtrada
# =========================================================
df_inmet = pd.read_excel("/content/04_BASE_MENSAL_LONGA_ESTUDO.xlsx")
df_inmet.columns = df_inmet.columns.str.strip().str.replace(" ","_")
df_inmet.rename(columns={"Longitude":"Long","Latitude":"Lat","Precipitacao":"Prec_INMET"}, inplace=True)

# =========================================================
# 4) Ler resumo das estações válidas
# =========================================================
resumo_estudo = pd.read_excel("/content/06_RESUMO_ESTACOES_ESTUDO.xlsx")
resumo_estudo.columns = resumo_estudo.columns.str.strip().str.replace(" ","_")
resumo_estudo.rename(columns={"Longitude":"Long","Latitude":"Lat"}, inplace=True)

# =========================================================
# 5) Selecionar estações com dados ≥ LIMITE_PERCENTUAL %
# =========================================================
estacoes_validas = resumo_estudo[resumo_estudo["Porcentagem_Concluida"] >= LIMITE_PERCENTUAL]
print(f"✅ Estaçoes com ≥{LIMITE_PERCENTUAL}% de dados: {len(estacoes_validas)}")

df_inmet_filtrado = df_inmet[df_inmet["Nome"].isin(estacoes_validas["Nome"])]

# =========================================================
# 6) Criar FeatureCollection para GEE
# =========================================================
fc_list = [
    ee.Feature(ee.Geometry.Point([row["Long"], row["Lat"]]), {"Nome": row["Nome"]})
    for idx, row in estacoes_validas[["Nome","Lat","Long"]].drop_duplicates().iterrows()
]
fc = ee.FeatureCollection(fc_list)

# =========================================================
# 7) Função para extrair ERA5 mensal acumulado (mm)
# =========================================================
def extrair_era5_mensal(fc, ano_ini, ano_fim):
    anos = range(ano_ini, ano_fim+1)
    meses = range(1,13)

    colecao = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select('total_precipitation_sum')
    colecao = colecao.map(lambda img: img.multiply(1000).copyProperties(img,['system:time_start']))

    dados_era5 = []

    for ano in anos:
        for mes in meses:
            inicio = ee.Date.fromYMD(ano, mes, 1)
            fim = inicio.advance(1, 'month')
            img_mes = colecao.filterDate(inicio, fim).sum()
            reduc = img_mes.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=5500)
            feats = reduc.getInfo().get("features", [])
            for f in feats:
                props = f.get("properties", {})
                nome = props.get("Nome","")
                val = props.get("mean", np.nan)
                dados_era5.append([nome, ano, mes, val])
    return pd.DataFrame(dados_era5, columns=["Nome","Ano","Mes","Prec_ERA5"])

# =========================================================
# 8) Extrair dados ERA5
# =========================================================
print("⏳ Extraindo dados ERA5 acumulados (pode demorar)...")
df_era5 = extrair_era5_mensal(fc, ANO_INICIAL, ANO_FINAL)
print("✅ Dados ERA5 extraídos")

# =========================================================
# 9) Combinar INMET + ERA5
# =========================================================
df_final = pd.merge(df_inmet_filtrado, df_era5, on=["Nome","Ano","Mes"], how="left")
df_final = df_final[["Nome","Lat","Long","Ano","Mes","Prec_INMET","Prec_ERA5"]]

df_final.to_excel("07_INMET_ERA5_MENSAL_ACUM.xlsx", index=False)
files.download("07_INMET_ERA5_MENSAL_ACUM.xlsx")
print("✅ Planilha final INMET + ERA5 acumulada exportada")
